In [1]:
import pandas as pd
import pickle
import os, re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import importlib
from util import ComBat
importlib.reload(ComBat)

from util import YuGene
importlib.reload(YuGene)

from util import CuBlock
importlib.reload(CuBlock)

from util import TDM
importlib.reload(TDM)

from util import QN
importlib.reload(QN)

<module 'util.QN' from '/main/projects/GANomics/trans_gan_project/util/QN.py'>

In [3]:
def generate_data(df_ag, df_ngs, test_index):
    test_ag = df_ag.loc[test_index,:]
    test_ngs = df_ngs.loc[test_index,:]
    train_ag = df_ag.loc[~df_ag.index.isin(test_index),:]
    train_ngs = df_ngs.loc[~df_ngs.index.isin(test_index),:]
    
    return train_ag, test_ag, train_ngs, test_ngs

### Load Dataset

In [ ]:
#BRCA2
exp='BRCA2'
ag_file = 'datasets/TCGA/BRCA/BRCA2_ag.tsv'
rs_file = 'datasets/TCGA/BRCA/BRCA2_rs.tsv'

#NCI60 (ag1)
exp='NCI60'
ag_file = 'datasets/NCI60/df_ag.xlsx'
rs_file = 'datasets/NCI60/df_ngs.xlsx'

#METSIM
exp='METSIM'
ag_file = 'datasets/METSIM/df_ag.tsv'
rs_file = 'datasets/METSIM/df_ngs.tsv'


In [39]:
exp='LAML2'
ag_file = 'datasets/TCGA/LAML/LAML2_ag.tsv'
rs_file = 'datasets/TCGA/LAML/LAML2_rs.tsv'

In [40]:
if ag_file.endswith('tsv'):
    df_ag = pd.read_csv(ag_file,sep="\t", index_col=0)
if ag_file.endswith('csv'):
    df_ag = pd.read_csv(ag_file, index_col=0).transpose() # Only for NB
elif ag_file.endswith('xlsx'):
    df_ag = pd.read_excel(ag_file, index_col=0)

if ag_file.endswith('tsv'):
    df_ngs = pd.read_csv(rs_file,sep="\t", index_col=0)
if ag_file.endswith('csv'):
    df_ngs = pd.read_csv(rs_file, index_col=0).transpose() # Only for NB
elif ag_file.endswith('xlsx'):
    df_ngs = pd.read_excel(rs_file,index_col=0)

print(df_ag.shape, df_ngs.shape)

(166, 17906) (166, 17906)


In [41]:
#GANomics-50
ep=5000
reps = [0,]
for rep in reps:
    for size in [50, ]:
        source_folder = f'{exp}/{exp}_{size}_ep{ep}_{rep}'
        file_name = f'{exp}-{size}-{rep}-NEW.pkl'
                
        df_ma_real = pd.read_csv(f'./results/{source_folder}/Microarray_real.csv',index_col=0)
        df_ma_fake = pd.read_csv(f'./results/{source_folder}/Microarray_fake.csv',index_col=0)
        df_rs_real = pd.read_csv(f'./results/{source_folder}/RNAseq_real.csv',index_col=0)
        df_rs_fake = pd.read_csv(f'./results/{source_folder}/RNAseq_fake.csv',index_col=0)
    
        test_index = df_ma_real.index
        train_ag, test_ag, train_ngs, test_ngs = generate_data(df_ag, df_ngs, test_index)
        # set gene name to the same
        train_ag.columns = list(range(train_ag.shape[1]))
        test_ag.columns = list(range(test_ag.shape[1]))
        train_ngs.columns = list(range(train_ngs.shape[1]))
        test_ngs.columns = list(range(test_ngs.shape[1]))
    
        result_combat = ComBat.example_usage_combat(train_ag, test_ag, train_ngs, test_ngs)
        df_ma_combat = result_combat['rnaseq_to_microarray']
        df_rs_combat = result_combat['microarray_to_rnaseq']
    
        result_yugene = YuGene.example_usage_yugene(train_ag, test_ag, train_ngs, test_ngs)
        df_ma_yugene = result_yugene['rnaseq_to_microarray']
        df_rs_yugene = result_yugene['microarray_to_rnaseq']
    
        ## CuBlock
        translator = CuBlock.fit_cublock_translator(
            source_train_df = train_ag,     # e.g., microarray (log2)
            target_train_df = train_ngs,    # e.g., RNA-seq (log2 CPM)
            k=5, n_repetitions=30, random_state=0
        )
        df_rs_cublock = CuBlock.translate_cublock(test_ag, translator)
        translator_rev = CuBlock.fit_cublock_translator(train_ngs, train_ag, k=5, n_repetitions=30)
        df_ma_cublock = CuBlock.translate_cublock(test_ngs, translator_rev)
    
            # --- TDM ---
        result_tdm = TDM.example_usage_tdm(train_ag, test_ag, train_ngs, test_ngs, data_is_log2=True)
        df_ma_tdm = result_tdm['rnaseq_to_microarray']
        df_rs_tdm = result_tdm['microarray_to_rnaseq']
    
        # --- Quantile Normalization (use-target) ---
        result_quantile = QN.example_usage_quantile(train_ag, test_ag, train_ngs, test_ngs)
        df_ma_quantile = result_quantile['rnaseq_to_microarray']
        df_rs_quantile = result_quantile['microarray_to_rnaseq']
    
        # Add to your sync dicts
        df_ma_sync = {
            'GANomics': df_ma_fake,
            'ComBat':   df_ma_combat,
            'YuGene':   df_ma_yugene,
            'CuBlock':  df_ma_cublock,
            'TDM':      df_ma_tdm,
            'Quantile': df_ma_quantile,
            'Baseline': df_rs_real,     # as in your current code
        }
        df_rs_sync = {
            'GANomics': df_rs_fake,
            'ComBat':   df_rs_combat,
            'YuGene':   df_rs_yugene,
            'CuBlock':  df_rs_cublock,
            'TDM':      df_rs_tdm,
            'Quantile': df_rs_quantile,
            'Baseline': df_ma_real,
        }
        
        pickle.dump({'df_ma_real':df_ma_real, 'df_ma_sync':df_ma_sync, 'df_rs_real':df_rs_real, 'df_rs_sync':df_rs_sync}, open(f'results/Comparison/{file_name}','wb'), )

Training: (50, 17906)
Testing: (116, 17906)
=== ComBat Cross-Platform Evaluation ===

1. Training ComBat model...
Training ComBat with 50 samples and 17906 genes

2. Transforming test data...
   Microarray -> RNA-seq...
Transforming 116 samples from microarray to rnaseq
   RNA-seq -> Microarray...
Transforming 116 samples from rnaseq to microarray
YuGene Corrected Implementation:
Training: MA (50, 17906), RNA (50, 17906)
Testing: MA (116, 17906), RNA (116, 17906)
=== YuGene Cross-Platform Evaluation (Corrected) ===
Training data: MA (50, 17906), RNA (50, 17906)
Test data: MA (116, 17906), RNA (116, 17906)

1. Training YuGene model...
Training YuGene with 50 samples and 17906 genes
YuGene model trained successfully
MA stats - mean: 0.5000, std: 0.2887
RNA stats - mean: 0.5000, std: 0.2881

2. Transforming test data...
   Applying YuGene to microarray test data...
Transforming 116 samples from microarray to yugene using YuGene
YuGene transformation applied to 116 samples
   Applying YuGe

## Evaluation

In [4]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import pickle

def measure_perf(df_real: pd.DataFrame,
                 df_fake: pd.DataFrame,
                 sample_col_name: str = "sample_id",
                 round_ndigits: int = 3) -> pd.DataFrame:
    """
    Returns a per-row metrics table with:
      sample_id, n_eval, L1, MAE, L2, RMSE, Pearson, Spearman
    Also sets the columns' name to 'metric' for clearer headers.
    """
    if df_real.shape != df_fake.shape:
        raise ValueError(f"Shape mismatch: real {df_real.shape} vs fake {df_fake.shape}")

    # Preserve original row labels for the sample column
    sample_ids = df_real.index.to_list()

    # Align and coerce to numeric
    A = df_real.reset_index(drop=True).apply(pd.to_numeric, errors="coerce")
    B = df_fake.reset_index(drop=True).apply(pd.to_numeric, errors="coerce")

    rows = []
    for i in range(A.shape[0]):
        a = A.iloc[i].to_numpy(dtype=float, copy=False)
        b = B.iloc[i].to_numpy(dtype=float, copy=False)

        m = np.isfinite(a) & np.isfinite(b)
        a, b = a[m], b[m]
        n = a.size

        if n == 0:
            row = {
                sample_col_name: sample_ids[i],
                "n_eval": 0,
                "Pearson": np.nan, "Spearman": np.nan,
                "L1": np.nan, "L2": np.nan, "MAE": np.nan, "RMSE": np.nan
                
            }
        else:
            diff = a - b
            L1 = np.sum(np.abs(diff))
            L2 = np.sqrt(np.sum(diff ** 2))
            MAE = L1 / n
            RMSE = L2 / np.sqrt(n)

            pearson = np.corrcoef(a, b)[0, 1] if (np.std(a) > 0 and np.std(b) > 0) else np.nan
            spearman = spearmanr(a, b, nan_policy="omit").correlation if n > 1 else np.nan
            
            row = {
                sample_col_name: sample_ids[i],
                "n_eval": int(n),
                "Pearson": pearson, "Spearman": spearman,
                "L1": L1, "L2": L2, "MAE": MAE, "RMSE": RMSE,
                
            }

        if round_ndigits is not None:
            for k in ("Pearson","Spearman","L1","L2","MAE","RMSE"):
                if pd.notna(row[k]):
                    row[k] = np.round(row[k], round_ndigits)

        rows.append(row)

    # Build DataFrame with a clear column order
    col_order = [sample_col_name, "n_eval", "Pearson", "Spearman", "L1",  "L2", "MAE", "RMSE"]
    out = pd.DataFrame(rows)[col_order]

    # Name the columns axis for nicer display in some renderers
    out.columns.name = "metric"
    return out

# Format "mean +- std" for a 1D numeric array, skipping NaNs/Infs
def _format_mean_std(arr, ndigits=2, pm_symbol="±"):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return "NA"
    mean = arr.mean()
    sd = arr.std(ddof=1) if arr.size > 1 else 0.0
    return f"{mean:.{ndigits}f} ({pm_symbol}{sd:.{ndigits}f})"

def _format_mean_ci(arr,
                    ndigits=2,
                    alpha=0.05,
                    ci_method="bootstrap",   # "bootstrap" (percentile) or "t"
                    n_boot=5000,
                    seed=0,
                    style="paren"):          # "paren" -> 'mean (95% CI lo–hi)'; "bracket" -> 'mean [lo, hi]'
    """
    Format mean with a 95% CI over the 1D array 'arr' (finite values only).
    Bootstrap CI is recommended for correlations and non-normal metrics.
    """
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return "NA"

    mean = float(a.mean())

    # Degenerate case: single observation -> CI collapses to the mean
    if a.size == 1:
        lo = hi = mean
    else:
        if ci_method.lower() == "bootstrap":
            # Percentile bootstrap on the mean
            rng = np.random.default_rng(seed)
            n = a.size
            idx = rng.integers(0, n, size=(int(n_boot), n))
            boot_means = a[idx].mean(axis=1)
            lo, hi = np.percentile(boot_means, [100*alpha/2, 100*(1 - alpha/2)])
        elif ci_method.lower() == "t":
            n = a.size
            sd = a.std(ddof=1)
            se = sd / np.sqrt(n) if n > 0 else np.nan
            crit = t.ppf(1 - alpha/2, df=n-1)
            lo, hi = mean - crit * se, mean + crit * se
        else:
            raise ValueError("ci_method must be 'bootstrap' or 't'")

    if style == "paren":
        return f"{mean:.{ndigits}f} (95% CI {lo:.{ndigits}f}–{hi:.{ndigits}f})"
    elif style == "bracket":
        return f"{mean:.{ndigits}f} [{lo:.{ndigits}f}, {hi:.{ndigits}f}]"
    else:
        return f"{mean:.{ndigits}f} (95% CI {lo:.{ndigits}f}–{hi:.{ndigits}f})"

def summarize_pairs_as_table(pairs,
                             labels,
                             ndigits=2,
                             metrics=("Pearson","Spearman","L1","L2","MAE","RMSE"),
                             alpha=0.05,
                             ci_method="bootstrap",
                             n_boot=5000,
                             seed=0,
                             style="paren"):
    """
    pairs:  list of (df_real, df_pred)
    labels: list of row labels matching the pairs
    Returns a DataFrame whose cells are 'mean (95% CI lo–hi)' for each metric.
    """
    if len(pairs) != len(labels):
        raise ValueError("pairs and labels must have the same length")

    rows = []
    for (df_real, df_pred), lbl in zip(pairs, labels):
        # Keep full precision here, no rounding in measure_perf
        per_row = measure_perf(df_real, df_pred, round_ndigits=None)

        row = {}
        for m in metrics:
            if m in per_row.columns:
                vals = per_row[m].to_numpy()
                row[m] = _format_mean_ci(vals,
                                         ndigits=ndigits,
                                         alpha=alpha,
                                         ci_method=ci_method,
                                         n_boot=n_boot,
                                         seed=seed,
                                         style=style)
            else:
                row[m] = "NA"
        rows.append(row)

    out = pd.DataFrame(rows, index=labels, columns=list(metrics))
    out.index.name = None
    out.columns.name = "Algorithm"
    return out

In [7]:
import pickle
import numpy as np
import pandas as pd

final_table=pd.DataFrame()
exps=['BRCA1','BRCA2','LUSC1','LAML1','NCI60','NCI602','NCI603','METSIM']
for exp in exps:
    for size in [50, ]:
        if 'BRCA' not in exp:
            file_name = f'{exp}-{size}-0-NEW.pkl'
        else:
            file_name = f'{exp}-100-0-NEW.pkl'
        data = pickle.load(open(f'results/Comparison/{file_name}','rb'))

        df_ma_real, df_ma_sync, df_rs_real, df_rs_sync = data['df_ma_real'], data['df_ma_sync'], data['df_rs_real'], data['df_rs_sync']
        
        df_ma_fake, df_ma_combat, df_ma_yugene, df_ma_cublock, df_ma_tdm, df_ma_qn = df_ma_sync['GANomics'], df_ma_sync['ComBat'], df_ma_sync['YuGene'], df_ma_sync['CuBlock'], df_ma_sync['TDM'], df_ma_sync['Quantile']
        df_rs_fake, df_rs_combat, df_rs_yugene, df_rs_cublock, df_rs_tdm, df_rs_qn = df_rs_sync['GANomics'], df_rs_sync['ComBat'], df_rs_sync['YuGene'], df_rs_sync['CuBlock'], df_rs_sync['TDM'], df_rs_sync['Quantile']
    
        pairs = [
            (df_ma_real, df_ma_fake),      # GANomics (MA)
            (df_ma_real, df_ma_combat),    # ComBat (MA)
            (df_ma_real, df_ma_yugene),    # YuGene (MA)
            (df_ma_real, df_ma_cublock),   # CuBlock (MA)
            (df_ma_real, df_ma_tdm),    # TDM (MA)
            (df_ma_real, df_ma_qn),   # Quantile (MA)
            (df_rs_real, df_rs_fake),      # GANomics (RNA-seq)
            (df_rs_real, df_rs_combat),    # ComBat (RNA-seq)
            (df_rs_real, df_rs_yugene),    # YuGene (RNA-seq)
            (df_rs_real, df_rs_cublock),   # CuBlock (RNA-seq)
            (df_rs_real, df_rs_tdm),    # TDM (RNA-seq)
            (df_rs_real, df_rs_qn),   # Quantile (RNA-seq)
            (df_ma_real, df_rs_real),      # Baseline (paired MA vs RNA-seq)
        ]
        
        labels = [
            "GANomics (MA)",
            "ComBat (MA)",
            "YuGene (MA)",
            "CuBlock (MA)",
            "TDM (MA)",
            "Quantile (MA)",
            "GANomics (RS)",
            "ComBat (RS)",
            "YuGene (RS)",
            "CuBlock (RS)",
            "TDM (RS)",
            "Quantile (RS)",
            "Baseline (MA vs RS)"
        ]
        
        
        curr_table = summarize_pairs_as_table(pairs, labels, ndigits=2)
        curr_table['Dataset'] = f'{exp}-{size}'
    
        final_table = pd.concat((final_table, curr_table), axis=0)

In [8]:
final_table

Algorithm,Pearson,Spearman,L1,L2,MAE,RMSE,Dataset
GANomics (MA),0.86 (95% CI 0.85–0.86),0.80 (95% CI 0.79–0.81),7622.13 (95% CI 7479.89–7767.38),86.97 (95% CI 85.46–88.53),0.49 (95% CI 0.48–0.49),0.69 (95% CI 0.68–0.71),BRCA1-50
ComBat (MA),0.15 (95% CI 0.15–0.16),0.15 (95% CI 0.14–0.16),47766.66 (95% CI 47629.22–47905.58),456.14 (95% CI 455.08–457.21),3.04 (95% CI 3.03–3.05),3.64 (95% CI 3.63–3.65),BRCA1-50
YuGene (MA),0.27 (95% CI 0.27–0.28),0.24 (95% CI 0.23–0.24),16325.72 (95% CI 16231.31–16426.09),175.50 (95% CI 174.55–176.45),1.04 (95% CI 1.03–1.05),1.40 (95% CI 1.39–1.41),BRCA1-50
CuBlock (MA),-0.02 (95% CI -0.02–-0.01),0.84 (95% CI 0.83–0.85),344444475.10 (95% CI 230042832.57–482593764.88),328647046.38 (95% CI 214281184.49–466682899.19),21943.33 (95% CI 14655.21–30744.33),2623139.60 (95% CI 1710313.44–3724890.91),BRCA1-50
TDM (MA),0.14 (95% CI 0.13–0.14),0.24 (95% CI 0.23–0.24),146194.39 (95% CI 146107.20–146282.75),1179.11 (95% CI 1178.44–1179.79),9.31 (95% CI 9.31–9.32),9.41 (95% CI 9.41–9.42),BRCA1-50
...,...,...,...,...,...,...,...
YuGene (RS),0.22 (95% CI 0.22–0.22),0.21 (95% CI 0.21–0.22),13418.54 (95% CI 13345.13–13494.05),238.84 (95% CI 238.06–239.64),1.97 (95% CI 1.96–1.98),2.90 (95% CI 2.89–2.91),METSIM-50
CuBlock (RS),-0.00 (95% CI -0.00–-0.00),0.94 (95% CI 0.94–0.94),525236935762.26 (95% CI 96920197011.25–1091491...,524131689689.59 (95% CI 96190626385.18–1089960...,77229368.59 (95% CI 14250874.43–160489864.18),6355563044.38 (95% CI 1166396923.32–1321673587...,METSIM-50
TDM (RS),NA,NA,NA,NA,NA,NA,METSIM-50
Quantile (RS),0.01 (95% CI 0.01–0.01),0.21 (95% CI 0.21–0.22),275033.16 (95% CI 275022.79–275043.97),31200.65 (95% CI 31200.52–31200.78),40.44 (95% CI 40.44–40.44),378.34 (95% CI 378.33–378.34),METSIM-50


In [9]:
final_table.to_excel('results/Comparison/Total_NEW_11142025.xlsx')

In [ ]:
final_table.reset_index(drop=False).sort_values(by=['index','Size'])

## Algorithm Comparison

In [ ]:
import pickle
import numpy as np
import pandas as pd

final_table=pd.DataFrame()
exp='BRCA'
for size in [10, 50]:
    file_name = f'{exp}-{size}.pkl'
    data = pickle.load(open(f'results/Comparison/{file_name}','rb'))
    df_ma_real, df_ma_sync, df_rs_real, df_rs_sync = data['df_ma_real'], data['df_ma_sync'], data['df_rs_real'], data['df_rs_sync']
    
    df_ma_fake, df_ma_combat, df_ma_yugene, df_ma_cublock, df_ma_tdm, df_ma_qn = df_ma_sync['GANomics'], df_ma_sync['ComBat'], df_ma_sync['YuGene'], df_ma_sync['CuBlock'], df_ma_sync['TDM'], df_ma_sync['Quantile']
    df_rs_fake, df_rs_combat, df_rs_yugene, df_rs_cublock, df_rs_tdm, df_rs_qn = df_rs_sync['GANomics'], df_rs_sync['ComBat'], df_rs_sync['YuGene'], df_rs_sync['CuBlock'], df_rs_sync['TDM'], df_rs_sync['Quantile']

    pairs = [
        (df_ma_real, df_ma_fake),      # GANomics (MA)
        (df_ma_real, df_ma_combat),    # ComBat (MA)
        (df_ma_real, df_ma_yugene),    # YuGene (MA)
        (df_ma_real, df_ma_cublock),   # CuBlock (MA)
        (df_ma_real, df_ma_tdm),    # TDM (MA)
        (df_ma_real, df_ma_qn),   # Quantile (MA)
        (df_rs_real, df_rs_fake),      # GANomics (RNA-seq)
        (df_rs_real, df_rs_combat),    # ComBat (RNA-seq)
        (df_rs_real, df_rs_yugene),    # YuGene (RNA-seq)
        (df_rs_real, df_rs_cublock),   # CuBlock (RNA-seq)
        (df_rs_real, df_rs_tdm),    # TDM (RNA-seq)
        (df_rs_real, df_rs_qn),   # Quantile (RNA-seq)
        (df_ma_real, df_rs_real),      # Baseline (paired MA vs RNA-seq)
    ]
    
    labels = [
        "GANomics (MA)",
        "ComBat (MA)",
        "YuGene (MA)",
        "CuBlock (MA)",
        "TDM (MA)",
        "Quantile (MA)",
        "GANomics (RS)",
        "ComBat (RS)",
        "YuGene (RS)",
        "CuBlock (RS)",
        "TDM (MA)",
        "Quantile (MA)",
        "Baseline (MA vs RS)"
    ]
        
    curr_table = summarize_pairs_as_table(pairs, labels, ndigits=2)
    curr_table['Size'] = f'GANomics-{size}'

    final_table = pd.concat((final_table, curr_table), axis=0)

In [ ]:
final_table